# 智能手套ASL识别 - 传统机器学习算法对比
# Traditional Machine Learning Algorithms Comparison

## 项目概述

本Notebook对比多种传统机器学习算法在ASL手势识别任务上的性能，包括：

### 对比算法
1. **支持向量机 (SVM)** - 最大化分类间隔的核方法
2. **K近邻 (KNN)** - 基于实例的懒惰学习
3. **XGBoost** - 极端梯度提升树
4. **LightGBM** - 微软轻量级梯度提升框架
5. **AdaBoost** - 自适应增强算法

### 数学基础

#### 1. 支持向量机 (SVM)

**目标函数**: 最大化分类间隔
```
min_{w,b,ξ}  1/2 ||w||² + C Σ ξ_i
s.t.         y_i(w·φ(x_i) + b) ≥ 1 - ξ_i
             ξ_i ≥ 0
```

**核函数**: K(x_i, x_j) = φ(x_i)·φ(x_j)
- 线性核: K(x,y) = x·y
- RBF核: K(x,y) = exp(-γ||x-y||²)
- 多项式核: K(x,y) = (γx·y + r)^d

**决策函数**: f(x) = sign(Σ α_i y_i K(x_i, x) + b)

#### 2. K近邻 (KNN)

**距离度量**:
- 欧氏距离: d(x,y) = √Σ(x_i - y_i)²
- 曼哈顿距离: d(x,y) = Σ|x_i - y_i|
- 闵可夫斯基距离: d(x,y) = (Σ|x_i - y_i|^p)^(1/p)

**预测**: y = mode({y_i | x_i ∈ N_k(x)})
其中 N_k(x) 是 x 的 k 个最近邻

#### 3. XGBoost

**目标函数**:
```
Obj = Σ L(y_i, ŷ_i) + Σ Ω(f_k)
     = Σ L(y_i, ŷ_i) + Σ (γT + 1/2λ||w||²)
```

其中:
- L: 损失函数 (logloss for classification)
- Ω: 正则化项
- T: 叶子节点数
- w: 叶子权重

**增益计算**:
```
Gain = 1/2 [ (G_L²/(H_L+λ)) + (G_R²/(H_R+λ)) - (G²/(H+λ)) ] - γ
```
其中 G = Σ g_i, H = Σ h_i 是梯度统计量

#### 4. LightGBM

**直方图算法**:
- 将连续特征离散化为 k 个bins
- 时间复杂度从 O(#data × #features) 降到 O(#bins × #features)

**叶子优先 (Leaf-wise) 生长**:
- 相比XGBoost的层级优先 (Level-wise)，选择分裂增益最大的叶子
- 更快收敛，但需要控制深度防止过拟合

## 第一步：环境配置与库导入

In [2]:
'''
Author: chouyougongchang233 megakidney@qq.com
Date: 2026-02-18 22:45:54
LastEditors: chouyougongchang233 megakidney@qq.com
LastEditTime: 2026-02-23 23:48:17
FilePath: \AlgeriaSmartGloves\Smart-glove-main\Traditional_ML_Comparison.ipynb
Description: 这是默认设置,请设置`customMade`, 打开koroFileHeader查看配置 进行设置: https://github.com/OBKoro1/koro1FileHeader/wiki/%E9%85%8D%E7%BD%AE
'''
# 安装必要的库
# !pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn

# 基础库
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


# scikit-learn 基础工具
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                           f1_score, precision_score, recall_score)

# 传统机器学习模型
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# 梯度提升库
import xgboost as xgb
import lightgbm as lgb

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
 
print("✅ 所有库导入成功！")
print(f"NumPy版本: {np.__version__}")
print(f"Pandas版本: {pd.__version__}")
print(f"XGBoost版本: {xgb.__version__}")
print(f"LightGBM版本: {lgb.__version__}")

✅ 所有库导入成功！
NumPy版本: 1.26.4
Pandas版本: 2.3.3
XGBoost版本: 2.1.4
LightGBM版本: 4.6.0


## 第二步：数据加载与预处理

In [5]:
# 配置参数
CONFIG = {
    'data_path': 'Smart-glove-main\modified dataset',
    'test_size': 0.3,
    'random_state': 42
}

# 手势类别
GESTURE_CLASSES = [
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
    'bad', 'deaf', 'fine', 'good', 'goodbye', 'hello', 
    'hungry', 'me', 'no', 'please'
]

def load_data(data_path, gesture_classes):
    """加载所有手势数据"""
    all_data = []
    
    for gesture in gesture_classes:
        file_path = os.path.join(data_path, f'{gesture}_merged.csv_exported.csv')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            # 选择特征列
            feature_cols = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
                          'GYRx', 'GYRy', 'GYRz', 'ACCx', 'ACCy', 'ACCz']
            df = df[['Alphabet'] + feature_cols].dropna()
            # 过滤负值
            df[feature_cols[:5]] = df[feature_cols[:5]].clip(lower=0)
            all_data.append(df)
            print(f"✅ 加载 {gesture}: {len(df)} 样本")
    
    combined_df = pd.concat(all_data, ignore_index=True)
    return combined_df

# 加载数据
df = load_data(CONFIG['data_path'], GESTURE_CLASSES)
print(f"\n📊 总计: {len(df)} 样本")
print(f"📊 类别数: {df['Alphabet'].nunique()}")

ValueError: No objects to concatenate

In [ ]:
# 数据预处理和特征工程
def preprocess_data(df):
    """
    数据预处理流程
    
    公式:
    1. 弯曲传感器归一化: x_norm = x / max(x)
    2. 加速度归一化: a_norm = a / 9.81 (转换为g)
    3. 陀螺仪归一化: g_norm = g / max(|g|)
    4. Z-score标准化: z = (x - μ) / σ
    """
    
    # 分离特征和标签
    feature_cols = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
                   'GYRx', 'GYRy', 'GYRz', 'ACCx', 'ACCy', 'ACCz']
    
    X = df[feature_cols].values
    y = df['Alphabet'].values
    
    # 特征归一化
    # 1. 弯曲传感器 (前5列)
    flex_max = X[:, :5].max()
    X[:, :5] = X[:, :5] / flex_max if flex_max > 0 else X[:, :5]
    
    # 2. 加速度计 (列5-7)
    X[:, 5:8] = X[:, 5:8] / 9.81
    
    # 3. 陀螺仪 (列8-10)
    gyro_max = np.abs(X[:, 8:11]).max()
    X[:, 8:11] = X[:, 8:11] / gyro_max if gyro_max > 0 else X[:, 8:11]
    
    # 标签编码
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # 划分数据集
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=CONFIG['test_size'], 
        random_state=CONFIG['random_state'], stratify=y_encoded
    )
    
    return X_train, X_test, y_train, y_test, le

# 执行预处理
X_train, X_test, y_train, y_test, label_encoder = preprocess_data(df)

print(f"\n📊 训练集: {X_train.shape[0]} 样本")
print(f"📊 测试集: {X_test.shape[0]} 样本")
print(f"📊 特征维度: {X_train.shape[1]}")
print(f"📊 类别数: {len(label_encoder.classes_)}")

## 第三步：定义评估函数

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, le, model_name):
    """
    综合模型评估函数
    
    评估指标:
    - Accuracy: (TP+TN)/(TP+TN+FP+FN)
    - Precision: TP/(TP+FP)
    - Recall: TP/(TP+FN)
    - F1-Score: 2*(Precision*Recall)/(Precision+Recall)
    - 交叉验证准确率
    """
    
    # 训练模型
    model.fit(X_train, y_train)
    
    # 预测
    y_pred = model.predict(X_test)
    
    # 计算指标
    results = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1_score': f1_score(y_test, y_pred, average='weighted')
    }
    
    # 交叉验证
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    results['cv_mean'] = cv_scores.mean()
    results['cv_std'] = cv_scores.std()
    
    print(f"\n{'='*60}")
    print(f"📊 {model_name} 评估结果")
    print(f"{'='*60}")
    print(f"准确率 (Accuracy):     {results['accuracy']:.4f}")
    print(f"精确率 (Precision):    {results['precision']:.4f}")
    print(f"召回率 (Recall):       {results['recall']:.4f}")
    print(f"F1分数:               {results['f1_score']:.4f}")
    print(f"交叉验证: {results['cv_mean']:.4f} (+/- {results['cv_std']*2:.4f})")
    
    return results, y_pred

def plot_confusion_matrix(y_true, y_pred, le, model_name):
    """绘制混淆矩阵"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'{model_name} - 混淆矩阵', fontsize=16)
    plt.xlabel('预测标签')
    plt.ylabel('真实标签')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{model_name.replace(" ", "_").lower()}.png', dpi=150)
    plt.show()

## 第四步：支持向量机 (SVM)

In [ ]:
# SVM 模型训练

print("\n" + "="*80)
print("🚀 支持向量机 (SVM) 训练")
print("="*80)

# SVM 参数说明:
# C: 正则化参数，控制误分类惩罚 (C越小，容错越多)
# kernel: 核函数 ('rbf', 'linear', 'poly')
# gamma: 核函数系数 (对于RBF核，gamma越大，单个样本影响范围越小)

svm_model = SVC(
    C=10.0,              # 正则化参数
    kernel='rbf',        # RBF核函数 K(x,y) = exp(-γ||x-y||²)
    gamma='scale',       # gamma = 1/(n_features * X.var())
    decision_function_shape='ovr',  # One-vs-Rest多分类
    random_state=42,
    verbose=True
)

svm_results, svm_pred = evaluate_model(
    svm_model, X_train, X_test, y_train, y_test, label_encoder, "SVM (RBF)")

In [ ]:
# SVM 不同核函数对比
print("\n📊 SVM 不同核函数对比")
print("-"*60)

kernels = ['linear', 'rbf', 'poly']
svm_kernel_results = []

for kernel in kernels:
    print(f"\n🔍 训练 SVM ({kernel}核)...")
    
    if kernel == 'poly':
        model = SVC(kernel=kernel, degree=3, C=1.0, random_state=42)
    else:
        model = SVC(kernel=kernel, C=10.0, random_state=42)
    
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    
    svm_kernel_results.append({'kernel': kernel, 'accuracy': acc})
    print(f"   准确率: {acc:.4f}")

# 可视化对比
plt.figure(figsize=(10, 6))
kernels_acc = [r['accuracy'] for r in svm_kernel_results]
plt.bar(kernels, kernels_acc, color=['skyblue', 'lightgreen', 'salmon'])
plt.xlabel('核函数类型')
plt.ylabel('准确率')
plt.title('SVM不同核函数性能对比')
plt.ylim(0.7, 1.0)
for i, v in enumerate(kernels_acc):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig('svm_kernels_comparison.png', dpi=150)
plt.show()

## 第五步：K近邻 (KNN)

In [ ]:
# KNN 模型训练

print("\n" + "="*80)
print("🚀 K近邻 (KNN) 训练")
print("="*80)

# KNN 参数说明:
# n_neighbors: K值，即考虑的邻居数量
# weights: 权重函数 ('uniform'=等权重, 'distance'=按距离倒数加权)
# metric: 距离度量 ('euclidean', 'manhattan', 'minkowski')
# p: Minkowski距离的幂次 (p=2为欧氏，p=1为曼哈顿)

knn_model = KNeighborsClassifier(
    n_neighbors=5,       # K=5
    weights='distance',  # 距离加权: w = 1/d
    metric='euclidean',  # 欧氏距离
    n_jobs=-1            # 并行计算
)

knn_results, knn_pred = evaluate_model(
    knn_model, X_train, X_test, y_train, y_test, label_encoder, "KNN (K=5)")

In [ ]:
# K值选择实验
print("\n📊 K值选择实验")
print("-"*60)

k_values = [1, 3, 5, 7, 9, 11, 15, 21]
knn_k_results = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k, weights='distance')
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    
    knn_k_results.append({
        'k': k, 
        'train_acc': train_acc, 
        'test_acc': test_acc
    })
    print(f"K={k:2d}: 训练准确率={train_acc:.4f}, 测试准确率={test_acc:.4f}")

# 可视化
plt.figure(figsize=(12, 6))
train_accs = [r['train_acc'] for r in knn_k_results]
test_accs = [r['test_acc'] for r in knn_k_results]

plt.plot(k_values, train_accs, 'o-', label='训练准确率', color='blue')
plt.plot(k_values, test_accs, 's-', label='测试准确率', color='red')
plt.xlabel('K值')
plt.ylabel('准确率')
plt.title('KNN不同K值性能对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('knn_k_selection.png', dpi=150)
plt.show()

## 第六步：XGBoost

In [ ]:
# XGBoost 模型训练

print("\n" + "="*80)
print("🚀 XGBoost 训练")
print("="*80)

# XGBoost 参数说明:
# n_estimators: 树的数量
# max_depth: 树的最大深度 (控制过拟合)
# learning_rate: 学习率 (每棵树的贡献权重)
# subsample: 每棵树的样本采样比例
# colsample_bytree: 每棵树的特征采样比例
# reg_alpha: L1正则化
# reg_lambda: L2正则化
# objective: 目标函数 ('multi:softprob' for multiclass)

num_classes = len(label_encoder.classes_)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,        # 树的数量
    max_depth=6,             # 最大深度
    learning_rate=0.1,       # 学习率
    subsample=0.8,           # 样本采样
    colsample_bytree=0.8,    # 特征采样
    reg_alpha=0.1,           # L1正则
    reg_lambda=1.0,          # L2正则
    objective='multi:softprob',
    num_class=num_classes,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

xgb_results, xgb_pred = evaluate_model(
    xgb_model, X_train, X_test, y_train, y_test, label_encoder, "XGBoost")

In [ ]:
# XGBoost 特征重要性
feature_names = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
                'GYRx', 'GYRy', 'GYRz', 'ACCx', 'ACCy', 'ACCz']

plt.figure(figsize=(12, 8))
importance_scores = xgb_model.feature_importances_
indices = np.argsort(importance_scores)[::-1]

plt.bar(range(len(importance_scores)), importance_scores[indices])
plt.xticks(range(len(importance_scores)), [feature_names[i] for i in indices], rotation=45)
plt.xlabel('特征')
plt.ylabel('重要性分数')
plt.title('XGBoost特征重要性')
plt.tight_layout()
plt.savefig('xgboost_feature_importance.png', dpi=150)
plt.show()

# 打印重要性排序
print("\n📊 XGBoost特征重要性排序:")
for i in indices:
    print(f"  {feature_names[i]:10s}: {importance_scores[i]:.4f}")

## 第七步：LightGBM

In [ ]:
# LightGBM 模型训练

print("\n" + "="*80)
print("🚀 LightGBM 训练")
print("="*80)

# LightGBM 参数说明:
# boosting_type: 'gbdt' (梯度提升树), 'dart' (Dropout+boosting)
# num_leaves: 叶子节点数 (控制复杂度，2^max_depth)
# max_depth: 树深度 (-1表示无限制)
# learning_rate: 学习率
# feature_fraction: 每轮特征采样比例
# bagging_fraction: 每轮数据采样比例
# bagging_freq: 采样频率
# min_child_samples: 叶子最小样本数
# reg_alpha/reg_lambda: L1/L2正则

lgb_model = lgb.LGBMClassifier(
    boosting_type='gbdt',     # 梯度提升树
    num_leaves=31,            # 叶子数
    max_depth=-1,             # 无限制
    learning_rate=0.1,        # 学习率
    n_estimators=200,         # 树的数量
    subsample=0.8,            # 数据采样
    colsample_bytree=0.8,     # 特征采样
    reg_alpha=0.1,            # L1正则
    reg_lambda=1.0,           # L2正则
    objective='multiclass',   # 多分类
    num_class=num_classes,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_results, lgb_pred = evaluate_model(
    lgb_model, X_train, X_test, y_train, y_test, label_encoder, "LightGBM")

In [ ]:
# LightGBM 训练曲线
from sklearn.model_selection import validation_curve

print("\n📊 LightGBM 迭代过程")
print("-"*60)

# 重新训练以获取eval结果
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_test, y_test, reference=lgb_train)

params = {
    'objective': 'multiclass',
    'num_class': num_classes,
    'metric': 'multi_logloss',
    'learning_rate': 0.1,
    'num_leaves': 31,
    'verbose': -1
}

evals_result = {}
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=200,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=[lgb.record_evaluation(evals_result)]
)

# 绘制训练曲线
plt.figure(figsize=(12, 6))
ax = lgb.plot_metric(evals_result, metric='multi_logloss')
plt.title('LightGBM训练过程')
plt.tight_layout()
plt.savefig('lightgbm_training_curve.png', dpi=150)
plt.show()

## 第八步：AdaBoost

In [ ]:
# AdaBoost 模型训练

print("\n" + "="*80)
print("🚀 AdaBoost 训练")
print("="*80)

# AdaBoost 数学原理:
# 1. 初始化样本权重: w_i = 1/N
# 2. 对于每个基学习器 t=1..T:
#    - 训练弱分类器 h_t(x)
#    - 计算加权错误率: ε_t = Σ w_i * I(y_i ≠ h_t(x_i))
#    - 计算分类器权重: α_t = 0.5 * ln((1-ε_t)/ε_t)
#    - 更新样本权重: w_i ← w_i * exp(-α_t * y_i * h_t(x_i))
#    - 归一化权重
# 3. 最终分类器: H(x) = sign(Σ α_t * h_t(x))

ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),  # 基学习器
    n_estimators=100,      # 迭代次数
    learning_rate=0.5,     # 学习率
    random_state=42
)

ada_results, ada_pred = evaluate_model(
    ada_model, X_train, X_test, y_train, y_test, label_encoder, "AdaBoost")

## 第九步：算法性能对比

In [ ]:
# 汇总所有结果
all_results = [
    svm_results,
    knn_results,
    xgb_results,
    lgb_results,
    ada_results
]

# 创建对比DataFrame
comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df.set_index('model_name')

print("\n" + "="*80)
print("📊 算法性能对比汇总")
print("="*80)
print(comparison_df.round(4))

# 保存结果
comparison_df.to_csv('ml_models_comparison.csv')
print("\n✅ 结果已保存到: ml_models_comparison.csv")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['准确率', '精确率', '召回率', 'F1分数']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    values = comparison_df[metric].values
    models = comparison_df.index
    
    bars = ax.bar(models, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    ax.set_ylabel(title)
    ax.set_ylim(0.7, 1.0)
    ax.set_title(f'{title}对比')
    ax.grid(axis='y', alpha=0.3)
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.savefig('ml_models_comparison.png', dpi=200)
plt.show()

In [ ]:
# 交叉验证对比
plt.figure(figsize=(12, 6))

cv_means = comparison_df['cv_mean'].values
cv_stds = comparison_df['cv_std'].values
models = comparison_df.index

x_pos = np.arange(len(models))
plt.bar(x_pos, cv_means, yerr=cv_stds, capsize=5, 
       color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'],
       alpha=0.7, edgecolor='black')

plt.xlabel('模型')
plt.ylabel('交叉验证准确率')
plt.title('5折交叉验证准确率对比 (含标准差)')
plt.xticks(x_pos, models)
plt.ylim(0.7, 1.0)
plt.grid(axis='y', alpha=0.3)

# 添加数值标签
for i, (mean, std) in enumerate(zip(cv_means, cv_stds)):
    plt.text(i, mean + std + 0.01, f'{mean:.3f}±{std:.3f}', 
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('cv_comparison.png', dpi=150)
plt.show()

## 第十步：总结与建议

In [ ]:
# 找出最佳模型
best_model = comparison_df['accuracy'].idxmax()
best_accuracy = comparison_df.loc[best_model, 'accuracy']

print("\n" + "="*80)
print("🏆 最佳模型推荐")
print("="*80)
print(f"\n最佳模型: {best_model}")
print(f"测试准确率: {best_accuracy:.4f}")

# 性能排序
print("\n📊 按准确率排序:")
sorted_models = comparison_df.sort_values('accuracy', ascending=False)
for idx, (model, row) in enumerate(sorted_models.iterrows(), 1):
    print(f"  {idx}. {model:15s}: {row['accuracy']:.4f}")

# 各模型优缺点
print("\n" + "="*80)
print("📋 各模型特点分析")
print("="*80)

analysis = {
    "SVM": {
        "优点": ["理论基础扎实", "泛化能力强", "适合高维数据"],
        "缺点": ["训练慢(大数据集)", "核函数选择困难", "内存消耗大"],
        "适用": "中小规模数据集，特征维度高"
    },
    "KNN": {
        "优点": ["简单直观", "无需训练", "适合多分类"],
        "缺点": ["预测慢", "对异常值敏感", "维度灾难"],
        "适用": "小规模数据集，实时性要求不高"
    },
    "XGBoost": {
        "优点": ["准确率高", "防止过拟合", "支持并行", "特征重要性"],
        "缺点": ["调参复杂", "内存占用大"],
        "适用": "大规模数据，准确率要求高"
    },
    "LightGBM": {
        "优点": ["训练速度快", "内存效率高", "准确率高", "支持大数据"],
        "缺点": ["可能过拟合", "对异常值敏感"],
        "适用": "大规模数据，需要快速训练"
    },
    "AdaBoost": {
        "优点": ["不易过拟合", "可解释性强", "通用性好"],
        "缺点": ["对异常值敏感", "训练速度慢"],
        "适用": "需要可解释性的场景"
    }
}

for model, info in analysis.items():
    print(f"\n{model}:")
    print(f"  优点: {', '.join(info['优点'])}")
    print(f"  缺点: {', '.join(info['缺点'])}")
    print(f"  适用: {info['适用']}")

print("\n" + "="*80)
print("💡 部署建议")
print("="*80)
print("""
1. **ESP32边缘部署**: 
   - 推荐: XGBoost (准确率最高) 或 LightGBM (速度快)
   - 注意: 需要将模型转换为C++代码或使用micromlgen

2. **云端/PC部署**: 
   - 推荐: XGBoost (综合性能最佳)
   - 备选: SVM (如果数据量小)

3. **实时性要求高**: 
   - 推荐: LightGBM 或简化版XGBoost
   - 减少树的深度和数量以加速推理

4. **模型大小敏感**: 
   - 推荐: SVM (模型小) 或 KNN (无模型)
   - XGBoost/LightGBM可通过剪枝减小体积
""")